In [1]:
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import json
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from pydantic import BaseModel
from collections import defaultdict

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_cohere import CohereRerank


/home/rajch/Document/Self rag/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
load_dotenv(dotenv_path=".env")
# print("API KEY:", os.getenv("OPENAI_API_KEY"))
# print("Sec API KEY:", os.getenv("COHERE_API_KEY"))

True

In [3]:
# Extract the contents from the documents
def document_partitioning(file_path:str):
    """Extrating Structured data from the Unstructured Documents"""

    print(f"Partitioning the contents of the file from {file_path}")

    elements = partition_pdf(
        filename = file_path,
        strategy = "hi_res",
        infer_table_structure=True,
        extract_image_block_types=['Image'],
        extract_image_block_to_payload=True
    )
    
    print(f"Partitioned elements : \n {elements}")
    
    return elements

In [4]:
elements = document_partitioning("Document/attention-is-all-you-need-Paper.pdf")

Partitioning the contents of the file from Document/attention-is-all-you-need-Paper.pdf


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


Partitioned elements : 
 [<unstructured.documents.elements.Title object at 0x7845326ac890>, <unstructured.documents.elements.NarrativeText object at 0x7845326afef0>, <unstructured.documents.elements.NarrativeText object at 0x7845243fd700>, <unstructured.documents.elements.Text object at 0x7845243fdb20>, <unstructured.documents.elements.NarrativeText object at 0x7845325855e0>, <unstructured.documents.elements.NarrativeText object at 0x7845243fdac0>, <unstructured.documents.elements.Text object at 0x784532587350>, <unstructured.documents.elements.NarrativeText object at 0x784532584920>, <unstructured.documents.elements.NarrativeText object at 0x7845325854c0>, <unstructured.documents.elements.Text object at 0x7845325869f0>, <unstructured.documents.elements.NarrativeText object at 0x784532586e10>, <unstructured.documents.elements.Text object at 0x784532585be0>, <unstructured.documents.elements.NarrativeText object at 0x7845325841d0>, <unstructured.documents.elements.Title object at 0x78453

In [5]:
## Chucking the extracted sturctured elements using title
def chunking_by_title(elements):
    
    print(f"\n\nChunking the elements using title method.\n")

    chunks = chunk_by_title(
        elements= elements,
        max_characters = 3000,
        new_after_n_chars = 2500,
        combine_text_under_n_chars = 500
    )
    
    print(f"Loading the chunk size : {len(chunks)}\n")

    return chunks

In [6]:
chunks = chunking_by_title(elements)
chunks



Chunking the elements using title method.

Loading the chunk size : 21



In [7]:
type(chunks[0])

unstructured.documents.elements.CompositeElement

In [8]:
chunks[0].to_dict()

{'type': 'CompositeElement',
 'element_id': '1963bf0a-8619-4e38-b824-dc399b1c53e7',
 'text': 'Attention Is All You Need\n\nAshish Vaswani∗ Google Brain avaswani@google.com\n\nNoam Shazeer∗ Google Brain noam@google.com\n\nNiki Parmar∗\n\nGoogle Research nikip@google.com\n\nJakob Uszkoreit∗ Google Research usz@google.com\n\nLlion Jones∗\n\nGoogle Research llion@google.com\n\nAidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu\n\nŁukasz Kaiser∗\n\nGoogle Brain lukaszkaiser@google.com\n\nIllia Polosukhin∗ ‡\n\nillia.polosukhin@gmail.com\n\nAbstract\n\nThe dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation ta

In [9]:
chunks[5].metadata.orig_elements[7].to_dict()

{'type': 'Image',
 'element_id': '0511dd20be587679e4a6e98ee867b0e2',
 'text': '',
 'metadata': {'coordinates': {'points': ((np.float64(850.5),
     np.float64(457.0659722222226)),
    (np.float64(850.5), np.float64(1075.8659722222224)),
    (np.float64(1162.0), np.float64(1075.8659722222224)),
    (np.float64(1162.0), np.float64(457.0659722222226))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-02-17T03:56:28',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 4,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': 'Document',
  'filename': 'attention-is-all-you-need-Paper.pdf'}}

In [10]:
def seperate_using_types(chunks):
    
    content_data={
        'text': chunks.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    if hasattr(chunks, 'metadata') and hasattr(chunks.metadata, 'orig_elements'):
        for elements in chunks.metadata.orig_elements:
            elements_type = type(elements).__name__
            
            if elements_type == 'Table':
                content_data['types'].append('table')
                table_content = getattr(elements.metadata, 'text_as_html', elements.text)
                content_data['tables'].append(table_content)
                
            elif elements_type == 'Image':
                if hasattr(chunks, 'metadata') and  hasattr(chunks.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(elements.metadata.image_base64)

    content_data['types'] = list(set(content_data['types']))
       
    return content_data


In [11]:
def summarize_content(text:str, tables:list[str], images:list[str])->str:
    """
    Summarize the contents of the table and images.
    """
    try:
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        
        #Creating a prompt for the llm
        prompt_text = f"""You are to create a searchable description for document content retrival
        
            CONTENT TO ANALYZE:
            TEXT:
            {text}
            
        """

        if tables:
            prompt_text += "Tables \n"
            
            for i, table in enumerate(tables):
               prompt_text += f"Table {i+1}: \n {table}\n\n" 
               
               prompt_text += """
               YOUR TASK:
               Generate a comprehensive, searchable description that covers:
               
               1. Key facts, numbers, and data points from text and tables
               2. Main topics and concepts discussed
               3. Questions this content could answer
               4. Visual content analysis (charts, diagrams, patterns in images)
               5. Alternative search terms users might use
               
               Make it detailed and serachable - prioritize findability over brevity
               
               SEARCHABLE DESCRIPTION:
               """

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]

        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
            
        message = HumanMessage(content=message_content)
            
        response = llm.invoke([message])
        
        return response.content
    except Exception as e:
        print(f" AI summary failed: {e}")
        #Fallback to simple summary
        summary = f"{text[:300]}"
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary    

In [12]:
def summarize_chunks(chunks):
    """
    Summarize the chunks which contain text, images, tables 
    """
    
    langchain_document = []
    
    # total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        # current_chunk = i+1
        
        #Analyze the chunk 
        analyze_chunk = seperate_using_types(chunk)
            
        if analyze_chunk['tables'] or analyze_chunk['images']:
            
            print(f"Creating summary for the tables and images")
            
            try:
                enhanced_summary = summarize_content(
                    analyze_chunk['text'],
                    analyze_chunk['tables'],
                    analyze_chunk['images']
                )
                print("AI summary was completed successfully.")

            except Exception as e:
                print(f"Summarizing failed {e}")
        
                enhanced_summary = analyze_chunk['text']

        else:
            print("No tables or image are found.")
            enhanced_summary = analyze_chunk['text']


        # Creating Langchain document
        doc = Document(
            page_content = enhanced_summary,
            metadata = {
                "original_content": json.dumps({
                    "raw_text": analyze_chunk['text'],
                    "tables_html": analyze_chunk['tables'],
                    "images_base64": analyze_chunk['images']
                })
            }
        )
        
        langchain_document.append(doc)
    
    print(f"Created langchain document of chunks size of {len(langchain_document)}.")

    return langchain_document

In [13]:
doc = summarize_chunks(chunks)

No tables or image are found.
No tables or image are found.
No tables or image are found.
No tables or image are found.
Creating summary for the tables and images
AI summary was completed successfully.
Creating summary for the tables and images
AI summary was completed successfully.
No tables or image are found.
No tables or image are found.
No tables or image are found.
No tables or image are found.
Creating summary for the tables and images
AI summary was completed successfully.
No tables or image are found.
No tables or image are found.
No tables or image are found.
Creating summary for the tables and images
AI summary was completed successfully.
No tables or image are found.
Creating summary for the tables and images
AI summary was completed successfully.
No tables or image are found.
No tables or image are found.
No tables or image are found.
No tables or image are found.
Created langchain document of chunks size of 21.


In [14]:
doc[0]

Document(metadata={'original_content': '{"raw_text": "Attention Is All You Need\\n\\nAshish Vaswani\\u2217 Google Brain avaswani@google.com\\n\\nNoam Shazeer\\u2217 Google Brain noam@google.com\\n\\nNiki Parmar\\u2217\\n\\nGoogle Research nikip@google.com\\n\\nJakob Uszkoreit\\u2217 Google Research usz@google.com\\n\\nLlion Jones\\u2217\\n\\nGoogle Research llion@google.com\\n\\nAidan N. Gomez\\u2217 \\u2020 University of Toronto aidan@cs.toronto.edu\\n\\n\\u0141ukasz Kaiser\\u2217\\n\\nGoogle Brain lukaszkaiser@google.com\\n\\nIllia Polosukhin\\u2217 \\u2021\\n\\nillia.polosukhin@gmail.com\\n\\nAbstract\\n\\nThe dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutio

In [15]:
#Adding unique ids to the chunks 
for i, chunk in enumerate(doc,1):
    source = chunk.metadata.get("source", "doc")
    chunk.metadata["id"] = f"{source}_chunk_{i}"

In [16]:
doc[0].metadata['id']

'doc_chunk_1'

In [17]:
# To Visualize the langchain document, im exporting it to a json file.
def export_docs_to_json(chunks, filename="chunks_export.json"):
    
    export_data=[]
    
    for i,chunk in enumerate(chunks):
        chunk_data={
            "chunk_id":i+1,
            "enhanced_content_summary": chunk.page_content,
            "metadata":{
                "original_content": json.loads(chunk.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    #create and save the json file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print("Json file created!")
    

export_docs_to_json(doc)

Json file created!


In [18]:
def convert_to_embeddings(chunks, persistant_directory="db/chroma.db"):
    """Convert the chunks into vector representation and store it in chroma.db"""
    
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    
    embeddings = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persistant_directory,
        collection_metadata={"hnsw:space":"cosine"}
    )
    
    print("Vector database creation completed!")
    
    return embeddings

vec_doc = convert_to_embeddings(doc)

Vector database creation completed!


In [19]:
#Pydantic model for structured output
class Query_Variation(BaseModel):
    queries: list[str]
    
    
    
#Original Query
original_query = "What is the need of Transformer and explain the architecture?"
print(f"Original query:'{original_query}'")



# Generate Multiple query variation of the same query
llm = ChatOpenAI(model="gpt-4o", temperature=0)

llm_with_tools = llm.with_structured_output(Query_Variation)

prompt = f"""Generate 3 different variation of the query that would help
retrieve relevant documents.

Original query: {original_query}

Return 3 alternative queries that rephrase or approach the same question with 
different angles.
"""

response = llm_with_tools.invoke(prompt)
query_variation = response.queries

print("Generate Query Variations")
for i, variation in enumerate(query_variation, 1):
    print(f"{i}, {variation}")
    
print("\n"+"="*60)

Original query:'What is the need of Transformer and explain the architecture?'
Generate Query Variations
1, Why are Transformers important in modern technology, and how is their architecture structured?
2, Can you describe the significance of Transformers and detail their architectural design?
3, What role do Transformers play in current applications, and what does their architecture look like?



/home/rajch/Document/Self rag/venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Query_Variation(queries=[...chitecture look like?']), input_type=Query_Variation])
  return self.__pydantic_serializer__.to_python(


In [20]:
# Search with each query variation and store the results

#Semantic Search / Dense Retrieval
vec_retriever = vec_doc.as_retriever(search_kwargs={'k':10})

#Keyword Search / Sparse Retrieval
bm25_retriever = BM25Retriever.from_documents(doc)
bm25_retriever.k = 10

dense_retrieval_results = []
sparse_retrieval_results = []

for i, query in enumerate(query_variation, 1):
    print(f"\n ________ Results for query {i}: {query}_______")
    
    dense_docs = vec_retriever.invoke(query)
    dense_retrieval_results.append(dense_docs)
    
    sparse_docs = bm25_retriever.invoke(query)
    sparse_retrieval_results.append(sparse_docs)
    
    print(f"Vector search Retrieved {len(dense_docs)} documents:\n")
    print(f"Keyword search Retrieved {len(sparse_docs)} documents:\n")
    
    for j, doc in enumerate(dense_docs, i):
        print(f"Dense Document {j}:")
        print(f"{doc.page_content[:150]}.......\n")
    
    print("-"*50)
    print("-"*50)
    
    for j, doc in enumerate(sparse_docs, i):
        print(f"Sparse Document {j}:")
        print(f"{doc.page_content[:150]}.......\n")
    
    print("-"*50)
    print("-"*50)
    
print("\n" + "=" * 60)
print("Multi-Query Retrival completed")


 ________ Results for query 1: Why are Transformers important in modern technology, and how is their architecture structured?_______
Vector search Retrieved 10 documents:

Keyword search Retrieved 10 documents:

Dense Document 1:
3 Model Architecture

Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29]. Here, the encoder maps an inp.......

Dense Document 2:
3 Model Architecture

Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29]. Here, the encoder maps an inp.......

Dense Document 3:
3 Model Architecture

Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29]. Here, the encoder maps an inp.......

Dense Document 4:
3 Model Architecture

Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29]. Here, the encoder maps an inp.......

Dense Document 5:
3 Model Architecture

Most competitive neural sequence transd

In [21]:
sparse_retrieval_results[0][4].page_content

'7 Conclusion\n\nIn this work, we presented the Transformer, the ﬁrst sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention.\n\nFor translation tasks, the Transformer can be trained signiﬁcantly faster than architectures based on recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014 English-to-French translation tasks, we achieve a new state of the art. In the former task our best model outperforms even all previously reported ensembles.\n\nWe are excited about the future of attention-based models and plan to apply them to other tasks. We plan to extend the Transformer to problems involving input and output modalities other than text and to investigate local, restricted attention mechanisms to efﬁciently handle large inputs and outputs such as images, audio and video. Making generation less sequential is another research goals of ours.\n\nT

Hybrid Retriever (combination)

In [22]:
print("Setting up Hybrid Retriever...")
hybrid_retriever = EnsembleRetriever(
    retrievers = [vec_retriever, bm25_retriever],
    weights=[0.5, 0.5] #equal weights to but vector and keyword search
)
print("Setup complete!!")




test_query = "purchase cost 7.5 billion" 

retrieved_chunks = hybrid_retriever.invoke(test_query)
# for i, doc in enumerate(retrieved_chunks, 1):
#     print(f"{i}. {doc.page_content}")
print(len(retrieved_chunks))



Setting up Hybrid Retriever...
Setup complete!!
10


Apply Cohere Reranking 

In [28]:
print("Cohere Reranking")
print("*"*60)

#Initialize Cohere reranker
reranker = CohereRerank(model="rerank-english-v3.0", top_n=5)

reranked_docs = reranker.compress_documents(retrieved_chunks, query)

# Show reranked results
for i, doc in enumerate(reranked_docs, 1):
    print(f"{i:2d}, {doc.page_content}")
    
print("Number of docs : ",len(reranked_docs))

Cohere Reranking
************************************************************
 1, In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for signiﬁcantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.

2 Background

The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU [20], ByteNet [15] and ConvS2S [8], all of which use convolutional neural networks as basic building block, computing hidden representations in parallel for all input and output positions. In these models, the number of operations required to relate signals from two arbitrary input or output positions grows in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes it more difﬁcu

In [39]:
combined_input = f"""Based on the Following documents, please answer this question: {original_query}

Docouments:
{chr(10).join([f"-{doc.page_content}" for doc in reranked_docs])}

Please provide a clear, helpful anser using only the information from these documents. 
"""

In [40]:
llm = ChatOpenAI(model="gpt-4o")

messages = [
    SystemMessage(content="You are a helpful assistant"),
    HumanMessage(content=combined_input),
]

final_result = llm.invoke(messages)
print(f'Generated Response: ')
print(final_result.content)

Generated Response: 
The Transformer is a model architecture designed for sequence transduction tasks, such as translation, which relies entirely on an attention mechanism instead of traditional recurrence-based methods like RNNs. This design choice allows for improved parallelization, resulting in significantly faster training times compared to models based on recurrent or convolutional layers.

The need for the Transformer model arises from the limitations of previous models in learning dependencies between distant positions. Models like the Extended Neural GPU, ByteNet, and ConvS2S, although capable of parallel computation, found it challenging to connect signals from distant positions due to the linear or logarithmic growth in the number of operations required. The Transformer, by contrast, reduces this to a constant number of operations through self-attention, while addressing the potential reduction in effective resolution via Multi-Head Attention.

### Transformer Architecture



In [24]:
# print(type(dense_part))
# print(type(dense_part[0]))

In [25]:
# print(dense_part[0].page_content[:100])

In [26]:
# def reciprocal_rank_fusion(chunk_list, k=60, verbose=True):
#     """
#     Apply Reciprocal Rank Fusion across multiple ranked lists.
#     Returns list of (chunk, rrf_score) tuples sorted descending by score.
#     """
#     if not chunk_list:
#         if verbose:
#             print("No input lists → returning empty")
#         return []

#     if verbose:
#         print("═" * 50)
#         print("Reciprocal Rank Fusion")
#         print("═" * 50)

#     rrf_scores = defaultdict(float)
#     unique_chunks = {}           # doc_id → original chunk object

#     for list_idx, ranked_chunks in enumerate(chunk_list, 1):
#         if verbose:
#             print(f"  Processing retrieval method {list_idx}  ({len(ranked_chunks)} items)")

#         for rank, chunk in enumerate(ranked_chunks, 1):
#             # Use consistent identifier
#             doc_id = chunk.page_content[50:]
#             if doc_id not in unique_chunks:
#                 unique_chunks[doc_id] = chunk

#             # Add RRF contribution from this rank
#             rrf_scores[doc_id] += 1.0 / (k + rank)

#     # Sort descending by fused score
#     sorted_results = sorted(
#         [(unique_chunks[doc_id], score) for doc_id, score in rrf_scores.items()],
#                                #  ↑↑↑↑↑↑  ← this was the bug: was 'content' but should be 'doc_id'
#         key=lambda x: x[1],
#         reverse=True
#     )

#     if verbose:
#         print(f"→ Unique chunks after fusion: {len(sorted_results)}")
#         if sorted_results:
#             print(f"  Top score   : {sorted_results[0][1]:.4f}")
#             print(f"  Bottom score: {sorted_results[-1][1]:.4f}")
#             # Optional: show preview of top item
#             top_content = sorted_results[0][0].page_content[:120].replace("\n", " ")
#             print(f"  Top preview : {top_content}...")
#         print("═" * 50)

#     return sorted_results